<a href="https://colab.research.google.com/github/tobiasllop/Tesis/blob/main/arca_padron_alcance4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏛️ ARCA SDK — Consulta Padrón Alcance 4

Este notebook permite consultar datos de un contribuyente en el Padrón de ARCA (Alcance 4) usando la SDK [`@arcasdk/core`](https://www.afipts.com).

> **Requisitos previos:**
> - Certificado digital (`.crt`) y clave privada (`.key`) generados en el portal de ARCA
> - CUIT autorizado para el servicio `ws_sr_padron_a4`
>
> Si todavía no tenés el certificado, seguí la guía oficial: https://www.afipts.com/tutorial/obtain-testing-certificate.html

## ⚙️ Paso 1 — Instalar Node.js

In [ ]:
# Instalar Node.js 20 en el entorno de Colab
!curl -fsSL https://deb.nodesource.com/setup_20.x | bash -
!apt-get install -y nodejs
!echo "Node version:" && node --version
!echo "NPM version:" && npm --version

2026-05-16 21:46:29 - Installing pre-requisites
Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:4 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:6 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:8 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,915 kB]
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:10 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,295 kB]
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:12 http://archive.ubuntu.com/ubuntu jammy-updates/restricted amd64 Packages [7,341 kB]
Get:13 http://archive.ubuntu.com/ubu

## 📦 Paso 2 — Crear proyecto e instalar la SDK

In [ ]:
%%bash
mkdir -p /content/arca_consulta
cd /content/arca_consulta
npm init -y
npm install @arcasdk/core
echo "✅ SDK instalada correctamente"

Wrote to /content/arca_consulta/package.json:

{
  "name": "arca_consulta",
  "version": "1.0.0",
  "main": "index.js",
  "scripts": {
    "test": "echo \"Error: no test specified\" && exit 1"
  },
  "keywords": [],
  "author": "",
  "license": "ISC",
  "description": ""
}




added 52 packages, and audited 53 packages in 4s

8 packages are looking for funding
  run `npm fund` for details

found 0 vulnerabilities
✅ SDK instalada correctamente


In [7]:
%%bash
CUIT="20455010366"
NOMBRE_EMPRESA="Tobias Llop"

mkdir -p /content/arca_consulta
cd /content/arca_consulta

openssl genrsa -out mi_clave.key 2048

openssl req -new -key mi_clave.key -out mi_solicitud.csr \
  -subj "/C=AR/O=$NOMBRE_EMPRESA/CN=CUIT $CUIT/serialNumber=CUIT $CUIT"

echo "✅ Archivos generados:"
ls -lh mi_clave.key mi_solicitud.csr

✅ Archivos generados:
-rw------- 1 root root 1.7K May 16 22:06 mi_clave.key
-rw-r--r-- 1 root root  985 May 16 22:06 mi_solicitud.csr


In [8]:
from google.colab import files
files.download("/content/arca_consulta/mi_solicitud.csr")
print("📩 Subí este archivo en el portal de ARCA para obtener el .crt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

📩 Subí este archivo en el portal de ARCA para obtener el .crt


In [14]:
from google.colab import files
import shutil

print("📄 Subí el .crt que te dio ARCA:")
uploaded = files.upload()
for fname in uploaded:
    shutil.move(fname, f"/content/arca_consulta/{fname}")
    CERT_FILE = fname
    print(f"✅ Certificado guardado: {fname}")

KEY_FILE = "mi_clave.key"
print(f"🔐 Clave privada: {KEY_FILE}")

📄 Subí el .crt que te dio ARCA:


Saving mi_app_arca_5f7f928a4db13d87.crt to mi_app_arca_5f7f928a4db13d87.crt
✅ Certificado guardado: mi_app_arca_5f7f928a4db13d87.crt
🔐 Clave privada: mi_clave.key


## ✏️ Paso 4 — Configurar credenciales y CUIT a consultar

Completá los valores en esta celda antes de continuar.

In [15]:
# ============================================================
# CONFIGURACIÓN — completá estos valores
# ============================================================

CUIT_CONSULTANTE = 20455010366      # Tu CUIT (el del certificado)
CUIT_A_CONSULTAR = 20455010366      # CUIT del contribuyente a consultar
PRODUCTION       = False            # False = testing/homologación | True = producción

# Nombres de los archivos que subiste en el Paso 3
# (si los subiste arriba, ya están guardados en CERT_FILE y KEY_FILE)
# Podés sobreescribirlos manualmente si es necesario:
CERT_FILE = "mi_app_arca_5f7f928a4db13d87.crt"
KEY_FILE  = "mi_clave.key"

print(f"CUIT consultante : {CUIT_CONSULTANTE}")
print(f"CUIT a consultar : {CUIT_A_CONSULTAR}")
print(f"Certificado      : {CERT_FILE}")
print(f"Clave privada    : {KEY_FILE}")
print(f"Producción       : {PRODUCTION}")

CUIT consultante : 20455010366
CUIT a consultar : 20455010366
Certificado      : mi_app_arca_5f7f928a4db13d87.crt
Clave privada    : mi_clave.key
Producción       : False


## 📝 Paso 5 — Generar el script TypeScript

In [16]:
script = f"""
import {{ Arca }} from "@arcasdk/core";
import * as fs from "fs";

const arca = new Arca({{
  cuit: {CUIT_CONSULTANTE},
  cert: fs.readFileSync("./{CERT_FILE}", "utf8"),
  key:  fs.readFileSync("./{KEY_FILE}",  "utf8"),
  production: {'true' if PRODUCTION else 'false'},
}});

const cuitAConsultar = {CUIT_A_CONSULTAR};

(async () => {{
  try {{
    console.log(`Consultando CUIT: ${{cuitAConsultar}}...\\n`);

    // Verificar estado del servidor
    const status = await arca.registerScopeFourService.getServerStatus();
    console.log("Estado del servidor:", JSON.stringify(status, null, 2));

    // Consultar datos del contribuyente
    const datos = await arca.registerScopeFourService.getTaxpayerDetails(cuitAConsultar);

    if (datos) {{
      console.log("\\n✅ Datos del contribuyente:");
      console.log(JSON.stringify(datos, null, 2));
    }} else {{
      console.log("⚠️  Contribuyente no encontrado.");
    }}
  }} catch (err: any) {{
    console.error("❌ Error:", err.message);
    process.exit(1);
  }}
}})();
"""

with open("/content/arca_consulta/consulta.mts", "w") as f:
    f.write(script)

print("✅ Script generado en /content/arca_consulta/consulta.mts")
print("\n--- Contenido del script ---")
print(script)

✅ Script generado en /content/arca_consulta/consulta.mts

--- Contenido del script ---

import { Arca } from "@arcasdk/core";
import * as fs from "fs";

const arca = new Arca({
  cuit: 20455010366,
  cert: fs.readFileSync("./mi_app_arca_5f7f928a4db13d87.crt", "utf8"),
  key:  fs.readFileSync("./mi_clave.key",  "utf8"),
  production: false,
});

const cuitAConsultar = 20455010366;

(async () => {
  try {
    console.log(`Consultando CUIT: ${cuitAConsultar}...\n`);

    // Verificar estado del servidor
    const status = await arca.registerScopeFourService.getServerStatus();
    console.log("Estado del servidor:", JSON.stringify(status, null, 2));

    // Consultar datos del contribuyente
    const datos = await arca.registerScopeFourService.getTaxpayerDetails(cuitAConsultar);

    if (datos) {
      console.log("\n✅ Datos del contribuyente:");
      console.log(JSON.stringify(datos, null, 2));
    } else {
      console.log("⚠️  Contribuyente no encontrado.");
    }
  } catch (err: any)

## 🚀 Paso 6 — Ejecutar la consulta

In [17]:
%%bash
cd /content/arca_consulta
npx tsx consulta.mts

Consultando CUIT: 20455010366...

Estado del servidor: {
  "appserver": "OK",
  "dbserver": "OK",
  "authserver": "OK"
}


❌ Error: ns1:cms.cert.untrusted: Certificado no emitido por AC de confianza: {"exceptionName":"gov.afip.desein.dvadac.sua.view.wsaa.LoginFault","hostname":"wsaaext1.homo.afip.gov.ar"}


CalledProcessError: Command 'b'cd /content/arca_consulta\nnpx tsx consulta.mts\n'' returned non-zero exit status 1.

## 🔄 (Opcional) Consultar múltiples CUITs en lote

In [ ]:
# Lista de CUITs a consultar
CUITS_A_CONSULTAR = [
    20111111111,
    27222222222,
    30333333333,
]

script_batch = f"""
import {{ Arca }} from "@arcasdk/core";
import * as fs from "fs";

const arca = new Arca({{
  cuit: {CUIT_CONSULTANTE},
  cert: fs.readFileSync("./{CERT_FILE}", "utf8"),
  key:  fs.readFileSync("./{KEY_FILE}",  "utf8"),
  production: {'true' if PRODUCTION else 'false'},
}});

const cuits = {CUITS_A_CONSULTAR};
const resultados: Record<number, any> = {{}};

(async () => {{
  for (const cuit of cuits) {{
    try {{
      const datos = await arca.registerScopeFourService.getTaxpayerDetails(cuit);
      resultados[cuit] = datos ?? "No encontrado";
      console.log(`✅ ${{cuit}}:`, JSON.stringify(datos, null, 2));
    }} catch (err: any) {{
      resultados[cuit] = {{ error: err.message }};
      console.error(`❌ ${{cuit}}:`, err.message);
    }}
  }}

  // Guardar resultados en JSON
  fs.writeFileSync("./resultados.json", JSON.stringify(resultados, null, 2), "utf8");
  console.log("\\n📁 Resultados guardados en resultados.json");
}})();
"""

with open("/content/arca_consulta/consulta_batch.mts", "w") as f:
    f.write(script_batch)

print("✅ Script batch generado. Ejecutá la siguiente celda para correrlo.")

In [ ]:
%%bash
cd /content/arca_consulta
npx tsx consulta_batch.mts

In [ ]:
# Descargar el archivo de resultados
from google.colab import files
files.download("/content/arca_consulta/resultados.json")

---
## 📚 Referencias

| Recurso | Link |
|---|---|
| Documentación SDK | https://www.afipts.com/services/consulta_padron_alcance_4.html |
| Manual ARCA (PDF) | http://www.arca.gob.ar/ws/ws_sr_padron_a4/manual_ws_sr_padron_a4_v1.1.pdf |
| Obtener certificado de testing | https://www.afipts.com/tutorial/obtain-testing-certificate.html |
| Repositorio de la SDK | https://github.com/ralcorta/arcasdk |